# **PROYEK UAS**

## Nama Proyek : **Melakukan klasifikasi time series pada dataset DucksAndGeese**

## A. **CRISP-DM** (Cross-Industry Standard Process for Data Mining)

1.**Business Understanding** (Pemahaman Bisnis)

1.1 Latar Belakang
Dataset DucksAndGeese berasal dari UEA Time Series Archive dan berisi sinyal time-series yang mewakili 5 (lima) spesies berbeda yang terdiri dari beberapa jenis bebek (2 Duck) dan beberapa jenis angsa (3 Geese).
Permasalahan: mengklasifikasikan sinyal tersebut ke kategori yang benar.

1.2 Tujuan Proyek
Membangun model time series classification yang dapat mengklasifikasikan sinyal audio menjadi 5 spesies burung:
- Black-bellied Whistling Duck
- Canadian Goose
- Greylag Goose
- Pink-footed Goose
- White-faced Whistling Duck
Ini adalah klasifikasi multiclass (5 kelas).

1.3 Manfaat
- Penggunaan dalam bioacoustic identification
- Sistem monitoring fauna otomatis
- Klasifikasi otomatis berbasis machine learning tanpa memerlukan pemeriksaan manual/intervensi manusia

1.4 Kriteria kesuksesan
Model dikatakan berhasil jika mencapai:
- Accuracy ≥ 85% untuk baseline
- F1-score per kelas ≥ 80%
- Aplikasi Streamlit berjalan lancar dan mampu menerima input serta menampilkan prediksi dan confidence score

2.**Data Understanding** (Pemahaman Data) 

2.1 Gambaran Umum Dataset
Dataset DucksAndGeese merupakan univariate time series audio yang digunakan untuk mengklasifikasikan lima spesies bebek/angsa berdasarkan bentuk gelombang suara. Dataset ini merupakan versi univariate dari dataset multivariate DuckDuckGeese.

Tabel Spesifikasi Dataset
| Properti                         | Nilai                                   |
|----------------------------------|-----------------------------------------|
| Train Size                       | 50                                      |
| Test Size                        | 50                                      |
| Total Instances                  | 100                                     |
| Time Series Length               | 236,784 data points per sample          |
| Number of Classes                | 5                                       |
| Dimensions                       | 1 (audio waveform univariate)           |
| Datatype                         | AUDIO (raw waveform)                    |
| Sample rate setelah preprocessing| 44,100 Hz                               |
| Sumber                           | www.xenocanto.com                       |

Tabel Label Kelas
| Label | Spesies                          | Jumlah |
|-------|----------------------------------|--------|
| 0     | Black-bellied Whistling Duck     | 20     |
| 1     | Canadian Goose                   | 20     |
| 2     | Greylag Goose                    | 20     |
| 3     | Pink-footed Goose                | 20     |
| 4     | White-faced Whistling Duck       | 20     |

2.2 Asal-usul Dataset
Dataset ini merupakan hasil kurasi dari audio asli yang diambil dari www.xenocanto.com, yaitu platform komunitas ornithology (suara burung).

Menurut dokumentasi:
- Rekaman suara berasal dari kategori kualitas A atau B → berarti suara cukup bersih.
- Karena beda-beda sampel rate (misalnya 48kHz, 96kHz, 22kHz), maka semua audio downsampled ke 44,100 Hz.
- Setelah itu, dilakukan truncation ke panjang 236,784 data point karena:
    - panjang aslinya bervariasi
    - panjang tersebut adalah “shortest series length” dari seluruh rekaman
- Dataset sudah menyediakan TRAIN (50) dan TEST (50). Untuk pengembangan, split ulang TRAIN menjadi train / validation (mis. 80/20) untuk tuning hyperparameter dan memilih model sebelum uji pada TEST final.

Ini berarti, setiap instance adalah rekaman audio berdurasi ± 5.37 detik (236,784 / 44,100). Dengan kalkulasi 236,784 samples / 44,100 Hz ≈ 5.37 detik audio.

2.3 Struktur Data di Dalam File .ts
Setelah membuka file TRAIN dan TEST:
- Format: TS format (UEA standard)
- Setiap baris berisi:
    v1, v2, v3, ..., v236784 : classLabel
- Data bertipe float (amplitudo audio).
    ex : 0.0012, -0.0031, 0.0042, ... , -0.0008 : 3. Artinya data tersebut adalah Pink-footed Goose.

2.4 Karakteristik Data
2.4.1 Univariate Long Sequence
- Panjang data sangat besar (236k points) → model deep learning cocok, tapi perlu:
    - Downsampling
    - Segmenting
- CNN/ResNet/FCN khusus TSC

2.4.2 Balanced Classes
Semua kelas masing-masing 20 sampel, sehingga:
- Tidak perlu balancing
- Micro-F1 = Macro-F1 ≈ akurat

2.4.3 High Variability
Karena suara burung:
- Bentuk gelombang sangat bervariasi
- Banyak noise lingkungan
- High-frequency chirp/whistle patterns

2.4.4 Data Challenges
- Ukuran input besar → memory heavy
- Harus distandardisasi (z-score)
- Perlu filtering (bandpass 300–8000 Hz)
- Potensial cropping / feature extraction (MFCC optional)

2.5 EDA (Exploratory Data Analysis)
2.5.1 Struktur Dataset
| File                   | Isi                              |
|------------------------|----------------------------------|
| DucksAndGeese_TRAIN.ts | 50 instance, univariate, labeled |
| DucksAndGeese_TEST.ts  | 50 instance, univariate, labeled |

2.5.2 Basic Stats
| Statistik              | Nilai                    |
|------------------------|--------------------------|
| Total sample           | 100                      |
| Panjang time series    | 236,784 point per sample |
| Channel                | 1 (mono)                 |
| Durasi estimasi        | 5.37 detik per sample    |

2.5.3 Distribusi Label
Semua kelas memiliki 20 sampel → balanced.
| Label | Nama Spesies                      | Jumlah |
|-------|-----------------------------------|--------|
| 0     | Black-bellied Whistling Duck      | 20     |
| 1     | Canadian Goose                    | 20     |
| 2     | Greylag Goose                     | 20     |
| 3     | Pink-footed Goose                 | 20     |
| 4     | White-faced Whistling Duck        | 20     |

2.5.4 Visualisasi Waveform
Tujuan EDA pada audio time series:
- melihat noise
- melihat pola amplitudo
- melihat perbedaan antar spesies
Untuk setiap kelas, ambil 1 contoh → plot waveform.

Insight yang diharapkan:
1. Canadian Goose → nada lebih rendah (gelombang besar & lambat)
2. Whistling Duck → suara nyaring (gelombang cepat & rapat)
3. Greylag Goose → tone cenderung clustering

2.5.5 Analisis Statistik per Kelas
Hitung:
- mean amplitude
- standard deviation
- zero crossing count
- RMS energy
ex :    | Kelas | Mean | Std  | ZeroCrossRate   | RMS |
        |-------|------|------|---------------- |-----|
        | 0     | …    | …    | …               | …   |

3.**Data Preparation** / Preprocessing

Karena time series audio mentah tidak efisien untuk ML klasik, langkah preprocessing wajib jelas.
3.1 Normalisasi Sinyal
Setiap sequence dinormalisasi:
    X_norm = (X - mean) / std
Gunanya:
- mengurangi perbedaan amplitude antar recording
- mempercepat training

3.2 Downsampling (Opsional Tapi Direkomendasikan)
Dari 44.1 kHz → 16 kHz atau 8 kHz
Manfaat:
- memperkecil data 3×
- mempercepat model
- tidak menghilangkan informasi signifikan (karena suara burung mostly < 10 kHz)

3.3 Feature Extraction (Ini inti pipeline cepat)
Ekstraksi fitur dari audio:
a. MFCC (Mel-frequency Cepstral Coefficients)
- paling populer untuk audio classification
- biasanya 13–40 coefficient

b. Zero Crossing Rate (ZCR)
- mencerminkan frekuensi suara
- duck whistling biasanya memiliki ZCR lebih besar

c. Spectral Centroid
- menentukan “kecerahan” suara

d. RMS Energy
- melihat power sinyal
Hasil akhirnya berupa tabel fitur seperti:
| mfcc1  | mfcc2  | …  | ZCR | centroid | RMS  | label  |
|--------|--------|----|-----|----------|------|--------|

3.4 Transformasi ke Domain Frekuensi
Untuk sinyal audio burung, representasi frekuensi lebih deskriptif dibanding waveform mentah.
Metode yang digunakan:
1. STFT (Short-Time Fourier Transform)
- menghasilkan spectrogram (time × frequency)
- visual representation yang sangat informatif
2. Log-Mel Spectrogram
- dasar untuk CNN audio
- cocok untuk perbedaan tonal antar spesies

3.5 Final Feature Set
Fitur yang dikumpulkan untuk model ML klasik:
+-------------------------+----------------------------------------------+
| Kelompok Fitur          | Penjelasan                                   |
+-------------------------+----------------------------------------------+
| MFCC (13–40)            | inti identifikasi suara                      |
| Spectral Centroid       | kecerahan suara                              |
| Spectral Bandwidth      | lebar frekuensi                              |
| Zero Crossing Rate      | jumlah transisi 0                            |
| RMS Energy              | power sinyal audio                           |
| Roll-off frequency      | energi kumulatif musik                       |
+-------------------------+----------------------------------------------+
Total fitur biasanya 40–60 per sample → cukup kecil untuk ML klasik seperti Random Forest.

3.6 Split Data
Dataset asli menyediakan:
- TRAIN = 50
- TEST = 50
Pada TRAIN dilakukan split ulang:
- 80% Train
- 20% Validation
Untuk hyperparameter tuning sebelum testing final.

4.**Modeling** (Pemodelan)
4.1 Model yang digunakan
a. Model Utama = Random Forest
- sangat stabil untuk dataset kecil (100 sampel)
- robust terhadap noisy features
- tidak sensitif terhadap feature scaling
- training cepat

b. Model Alternatif = XGBoost
- performa tinggi pada dataset kecil–menengah
- lebih kuat menangani struktur non-linear
- dapat outperform Random Forest pada banyak kasus audio tabular

c. Baseline model: KNN
- sebagai pembanding sederhana
- Bagus sebagai minimum viable classifier

4.2 Pipeline Model :
    Raw Audio → Preprocessing → Feature Extraction → ML Model → Prediksi Kelas

Pipeline yang tepat untuk dataset UEA time series berbasis audio karena dataset DucksAndGeese adalah raw waveform panjang (236k points), maka:
- Preprocessing → normalisasi, downsampling
- Feature extraction → MFCC, ZCR, Spectral features
- Model → RF, XGB, KNN

4.3 Output Model
- prediksi salah satu dari 5 kelas spesies burung (0–4 → nama spesies)
- confidence probability untuk UI Streamlit (soft voting atau predict_proba)
- Untuk Streamlit:
    - waveform plot
    - spectrogram plot
    - hasil prediksi utama
    - confidence chart

5.**Evaluation** (Evaluasi)

Menggunakan test set resmi (50 sample).
5.1 Metrik Evaluasi:
1. Accuracy ≥ 85%
2. Classification Report ;
- precision
- recall
- f1-score per kelas ≥ 80%
3. Confusion Matrix

5.2 Evaluasi Model
Langkah evaluasi:
- Train pada TRAIN (80%)
- Validasi di VAL (20%) untuk:
    - parameter tuning RF (trees, depth)
    - XGBoost tuning (eta, max_depth)
- Final test pada TEST (50 sampel)

5.3 Insight evaluasi yang ditulis
- apakah goose vs duck mudah dibedakan
- kelas mana yang sering tertukar
- misclassification karena:
    - overlap frekuensi
    - noise

6.**Deployment** (Streamlit)

6.1 Tampilan Streamlit
1. Input
- upload audio .wav
- Periksa : rate, length (truncation kalau perlu)
- model akan melakukan preprocessing otomatis (normalisasi, downsampling, ekstraksi MFCC)
- fitur diekstrak (MFCC, ZCR, dll)
- Deploy model yang telah ditraining (pickle)

2. Output
- waveform plot
- MFCC heatmap (opsional)
- prediksi spesies
- confidence score
- penjelasan model

3. Struktur Streamlit
- Home → deskripsi proyek
- EDA → plot waveform & tabel statistik
- Model → prediksi audio upload
- About Dataset

4. File Deployment
    /streamlit_app.py
    /model.pkl
    /preprocessing.py
    /feature_extraction.py